# Notebook 07 — Static Feature Table

**Purpose**: Collapse the loan-month panel to one row per `LOAN_ID` with the origination-time features needed for M1 (baseline) and M2 (climate-augmented).

**Inputs**
- `fl_loan_month_clean.parquet`
- `fl_default_labels.parquet`

**Output**
- `fl_static_features.parquet` — one row per loan, all origination-time variables + labels

**Method notes**
- Origination-time features are constant across the loan-month panel; take the earliest observation per loan.
- `ORIG_DATE` in Fannie Mae is `MMYYYY`; we extract `ORIG_YEAR` for the climate merge.
- `ZIP` in Fannie Mae is already 3-digit (privacy convention); store as zero-padded string.
- `ORIG_LTV` is retained here for descriptive statistics but will be dropped in M1 (collinear with `ORIG_CLTV`).


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("your/data/path/here")

IN_LM   = DATA_DIR / "fl_loan_month_clean.parquet"
IN_LBL  = DATA_DIR / "fl_default_labels.parquet"
OUT_STC = DATA_DIR / "fl_static_features.parquet"

assert IN_LM.exists() and IN_LBL.exists()


## 1. Load loan-month panel and labels

In [ ]:
df  = pd.read_parquet(IN_LM)
lbl = pd.read_parquet(IN_LBL)
print(f"Loan-month rows: {len(df):,}")
print(f"Unique loans   : {df['LOAN_ID'].nunique():,}")
print(f"Labels rows    : {len(lbl):,}")


## 2. Collapse to one row per LOAN_ID

In [ ]:
static_cols = [
    'ORIG_DATE',       # MMYYYY
    'ORIG_RATE',
    'ORIG_UPB',
    'ORIG_TERM',
    'ORIG_LTV',
    'ORIG_CLTV',
    'NUM_BORR',
    'DTI',
    'CSCORE_B',
    'FIRST_FLAG',
    'PURPOSE',
    'PROP',
    'NO_UNITS',
    'OCC_STAT',
    'STATE',
    'ZIP',             # 3-digit
    'MI_PCT',
    'CHANNEL',
]

static = (df.sort_values(['LOAN_ID', 'PERIOD_YYYYMM'])
            .groupby('LOAN_ID')[static_cols]
            .first()
            .reset_index())

print(f"Static rows: {len(static):,}")
static.head()


## 3. Derive fields for downstream use

In [ ]:
# ORIG_YEAR from MMYYYY
def orig_year(x):
    s = str(int(x)).zfill(6)
    return int(s[2:])

static['ORIG_YEAR'] = static['ORIG_DATE'].apply(orig_year)

# FIRST_FLAG as 0/1
static['FIRST_FLAG'] = (static['FIRST_FLAG'].astype(str).str.upper().str.strip() == 'Y').astype(int)

# MI_PCT: missing means no MI, so 0
static['MI_PCT'] = pd.to_numeric(static['MI_PCT'], errors='coerce').fillna(0)

# ZIP3 as zero-padded string for the crosswalk merge
static['ZIP3'] = static['ZIP'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(3).str[:3]

print(static[['ORIG_YEAR', 'ZIP3', 'FIRST_FLAG', 'MI_PCT']].head())
print(f"\nOrigination year distribution:\n{static['ORIG_YEAR'].value_counts().sort_index()}")


## 4. Merge labels in

In [ ]:
static = static.merge(lbl, on='LOAN_ID', how='left')

# Any loans without a label? (Shouldn't happen but check)
missing_lbl = static['default_180dpd'].isna().sum()
print(f"Loans missing label: {missing_lbl}")

print(f"\nOverall default rate: {static['default_180dpd'].mean():.4f}")
print(f"Overall forbearance rate: {static['covid_forbearance'].mean():.4f}")


## 5. Missingness check on features

In [ ]:
static['HAS_MI'] = (static['MI_PCT'] > 0).astype(int)

In [ ]:
na_summary = static.isna().sum()
na_summary = na_summary[na_summary > 0].sort_values(ascending=False)
if len(na_summary) == 0:
    print("No missing values in static feature table.")
else:
    print("Missing values:")
    print(na_summary)


## 6. Descriptive statistics of key numeric features

In [ ]:
num_cols = ['CSCORE_B', 'DTI', 'ORIG_LTV', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB', 'MI_PCT']
static[num_cols].describe().round(2)


## 7. Save

In [ ]:
static.to_parquet(OUT_STC, index=False)
print(f"Saved: {OUT_STC}")
print(f"Shape: {static.shape}")
